In [0]:
CREATE OR REPLACE VIEW smart_meters.gold.vw_tou_vs_std_savings AS
SELECT
    d.year,
    d.month,
    d.day,
    d.hour,
    h.acorn_group,
    h.lclid,
    
    f.energy_spent AS energy_spent,
    
    f.money_spent AS actual_money_spent_tou,
    (f.energy_spent * std.price) AS simulated_cost_if_std,

    f.money_spent - (f.energy_spent * std.price) AS savings

FROM smart_meters.gold.fact_consumption AS f

JOIN smart_meters.gold.dim_time AS d 
    ON f.dim_time_id = d.id

JOIN smart_meters.gold.dim_household AS h 
    ON f.dim_household_lclid = h.lclid

JOIN smart_meters.gold.dim_tariffs AS t 
    ON f.dim_tariff_sk = t.sk

JOIN smart_meters.gold.dim_tariffs AS std 
    ON std.id = 1 
    AND make_date(d.year, d.month, d.day) >= std.valid_from
    AND (make_date(d.year, d.month, d.day) <= std.valid_to OR std.valid_to IS NULL)

WHERE t.id IN (2, 3, 4)

ORDER BY 
    d.year, d.month, d.day, d.hour, 
    h.acorn_group;

CREATE OR REPLACE VIEW smart_meters.gold.vw_yearly_lclid_savings AS
SELECT
    year,
    lclid,
    acorn_group,
    SUM(savings) AS total_yearly_savings
FROM smart_meters.gold.vw_tou_vs_std_savings
GROUP BY year, lclid, acorn_group
ORDER BY year, total_yearly_savings DESC;

CREATE OR REPLACE VIEW smart_meters.gold.vw_yearly_lclid_savings_acorn_group AS
SELECT
    acorn_group,
    SUM(total_yearly_savings) AS total_savings_by_group,
    COUNT(DISTINCT lclid) AS num_households,
    COUNT(DISTINCT year) AS num_years,
    ROUND(AVG(total_yearly_savings), 2) AS avg_savings_per_household
FROM smart_meters.gold.vw_yearly_lclid_savings
GROUP BY acorn_group
ORDER BY total_savings_by_group DESC;

CREATE OR REPLACE VIEW smart_meters.gold.vw_top5_savers AS
SELECT 
    lclid,
    acorn_group,
    year,
    total_yearly_savings
FROM smart_meters.gold.vw_yearly_lclid_savings
ORDER BY total_yearly_savings DESC
LIMIT 5;

SELECT * FROM smart_meters.gold.vw_yearly_lclid_savings_acorn_group LIMIT 10